# MatchMind - StatsBomb Data Exploration

This notebook is the first data-analysis component of **MatchMind**, an AI-powered football analysis project that will progressively combine:

- football event analytics;
- data visualization;
- semantic search and embeddings;
- Retrieval-Augmented Generation (RAG);
- Large Language Models (LLMs);
- tool calling and agentic AI.

The notebook deliberately begins with **raw football data and deterministic analysis**, before introducing any LLM component. The goal is to understand exactly what information is available, how it is represented, and which questions can already be answered reliably with Python.

## Analysis scope

| Item | Value |
| --- | --- |
| Competition | 2022 FIFA World Cup |
| Match | Argentina vs France |
| Stage | Final |
| Date | 18 December 2022 |
| StatsBomb match ID | `3869685` |
| Data provider | StatsBomb Open Data |
| Main file used | Event data |

The match is a particularly useful case study because it contains regular time, extra time, substitutions, goals, a wide variety of attacking actions, and a penalty shootout.

## Learning objectives

By the end of this notebook, we should be able to:

1. understand the structure of StatsBomb JSON event data;
2. distinguish common fields from event-specific nested fields;
3. convert nested JSON data into tabular Pandas data;
4. isolate and analyze shots, goals, passes, and substitutions;
5. understand the meaning of `period`, `location`, and `xG`;
6. separate match events from penalty-shootout events;
7. build a first spatial football visualization from shot coordinates;
8. identify which future MatchMind questions should be answered with deterministic tools rather than an LLM.

## Why start from the raw JSON?

It would be possible to use a high-level library that directly returns ready-made DataFrames. That would be faster, but it would hide an important part of the data model.

For this first notebook, the workflow is intentionally explicit:

```text
StatsBomb JSON
      |
      v
Python lists and dictionaries
      |
      v
Nested event structure
      |
      v
Pandas DataFrame
      |
      v
Filtering and aggregation
      |
      v
Football statistics and visualizations
```

This raw-first approach gives us a clear mental model of the information that future retrieval, RAG, and agent tools will operate on.

## Data source and attribution

The data used in this project comes from **StatsBomb Open Data**:

https://github.com/statsbomb/open-data

StatsBomb asks users who publish or share analysis based on its open data to identify StatsBomb as the data source. The project should preserve that attribution in its public documentation.

## Reproducibility

The raw files are expected under:

```text
MatchMind/
└── data/
    └── raw/
        └── statsbomb/
```

The `data/raw/` directory should remain ignored by Git. The notebook and project code are versioned, while the source data can be downloaded separately.

## 1. Imports

Only lightweight data-analysis libraries are needed at this stage.

| Library | Purpose |
| --- | --- |
| `json` | Read the raw StatsBomb JSON files. |
| `collections.Counter` | Count event categories before introducing Pandas operations. |
| `pathlib.Path` | Build portable filesystem paths. |
| `pprint` | Display nested Python dictionaries more clearly. |
| `pandas` | Normalize, filter, group, and aggregate event data. |

Keeping the dependency set small makes each step easier to understand.

In [ ]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint

import pandas as pd

print(f"Pandas version: {pd.__version__}")

## 2. Project paths

A notebook should not depend on a hard-coded absolute path such as `/Users/.../MatchMind`.

Instead, the following cell detects whether Jupyter was launched from:

- the project root, `MatchMind/`; or
- the notebook directory, `MatchMind/notebooks/`.

The resulting `PROJECT_ROOT` is then used to construct all data paths.

### Expected source files

| File | Role in the project |
| --- | --- |
| `3869685_events.json` | Detailed event stream for Argentina vs France. This is the main file used in this notebook. |
| `3869685_lineups.json` | Team lineups and player information. It will become useful for player and formation analysis. |
| `world_cup_2022_matches.json` | Match-level metadata for the 2022 World Cup. |
| `3869685_360.json` | StatsBomb 360 contextual information for supported events. It is checked here but not analyzed yet. |

This separation is useful because event data, lineup data, match metadata, and 360 context serve different analytical purposes.

In [ ]:
cwd = Path.cwd()

if (cwd / "data" / "raw" / "statsbomb").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw" / "statsbomb").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate 'data/raw/statsbomb'. "
        "Start Jupyter from the MatchMind project or its notebooks directory."
    )

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "statsbomb"

EVENTS_FILE = DATA_DIR / "3869685_events.json"
LINEUPS_FILE = DATA_DIR / "3869685_lineups.json"
MATCHES_FILE = DATA_DIR / "world_cup_2022_matches.json"
THREE_SIXTY_FILE = DATA_DIR / "3869685_360.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 3. Validate the local data files

Before running any analysis, we verify which expected files are available.

This is a simple reproducibility check. If a required file is missing, it is better to detect the problem immediately than to encounter a less informative error later in the notebook.

Only the event file is strictly required for the analyses below. The other files are displayed because they will be used in later MatchMind stages.

In [ ]:
files = {
    "Events": EVENTS_FILE,
    "Lineups": LINEUPS_FILE,
    "Matches": MATCHES_FILE,
    "360 data": THREE_SIXTY_FILE,
}

for name, path in files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:<10} | {status:<7} | {path}")

## 4. Load the raw event data

The StatsBomb event file is stored as a JSON array. When Python loads this file with `json.load`, the JSON array becomes a Python `list`, and every event becomes a Python `dict`.

Conceptually:

```text
JSON file
  |
  v
list of match events
  |
  +-- event 1 -> dictionary
  +-- event 2 -> dictionary
  +-- event 3 -> dictionary
  +-- ...
```

An event is not restricted to a single flat set of columns. It can contain nested objects whose structure depends on the event type.

For example:

- a `Pass` event can contain a nested `pass` object;
- a `Shot` event can contain a nested `shot` object;
- a `Substitution` event can contain a nested `substitution` object.

Understanding this nested structure is essential before flattening the data with Pandas.

In [ ]:
if not EVENTS_FILE.exists():
    raise FileNotFoundError(f"Events file not found: {EVENTS_FILE}")

with open(EVENTS_FILE, "r", encoding="utf-8") as file:
    events = json.load(file)

print(f"Python object type: {type(events).__name__}")
print(f"Number of events: {len(events)}")

## 5. Inspect the structure of one event

Every event has a set of common fields describing **when**, **where**, **which team**, and **what type of action** occurred. Other fields appear only when they are relevant to that event.

Some important common fields are:

| Field | Meaning |
| --- | --- |
| `id` | Unique identifier for the event. |
| `index` | Sequential position of the event in the match event stream. |
| `period` | Match period in which the event occurred. |
| `timestamp` | Time within the current period. |
| `minute`, `second` | Match clock representation used for readable timing. |
| `type` | Event category, such as `Pass`, `Shot`, `Carry`, or `Pressure`. |
| `team` | Team performing the event. |
| `player` | Player performing the event, when applicable. |
| `possession` | Identifier of the possession sequence. |
| `possession_team` | Team considered to be in possession. |
| `play_pattern` | How the possession started, for example regular play or a set piece. |
| `location` | Spatial starting position of the event in StatsBomb coordinates. |
| `under_pressure` | Optional indicator that the acting player was under pressure. |

Not every field is present in every event. This is normal for event-based data.

In [ ]:
first_event = events[0]

print(f"First event type: {type(first_event).__name__}")
print("\nAvailable top-level keys:")
print(list(first_event.keys()))

print("\nFull first event:")
pprint(first_event)

## 6. Access nested event fields

Many StatsBomb attributes are represented as nested dictionaries rather than simple strings.

For example, the event type may look conceptually like:

```python
{
    "id": 30,
    "name": "Pass"
}
```

Therefore:

```python
event["type"]
```

returns the complete nested dictionary, while:

```python
event["type"]["name"]
```

returns only the readable name, such as `"Pass"`.

The same pattern is used for teams, players, outcomes, body parts, techniques, and many other categorical attributes.

In [ ]:
event = events[10]

print("Event type :", event["type"]["name"])
print("Team       :", event["team"]["name"])
print("Minute     :", event["minute"])
print("Second     :", event["second"])

### 6.1 Create a compact event representation

Printing a complete nested event is useful for schema inspection, but it is too verbose for reading a sequence of actions.

The helper function below extracts only four pieces of information:

- match time;
- team;
- event type;
- player.

The use of `.get()` makes the formatter more robust when optional fields such as `player` are absent.

In [ ]:
def format_event(event):
    team = event.get("team", {}).get("name", "Unknown")
    player = event.get("player", {}).get("name", "N/A")
    event_type = event.get("type", {}).get("name", "Unknown")
    minute = event.get("minute", 0)
    second = event.get("second", 0)

    return (
        f"{minute:>3}:{second:02d} | "
        f"{team:<10} | "
        f"{event_type:<20} | "
        f"{player}"
    )

for event in events[:20]:
    print(format_event(event))

## 7. Explore event types

Before selecting a specific football action, it is useful to inspect the categories that actually occur in the match.

An event dataset contains much more than goals and passes. Depending on the match, it can include actions such as carries, pressures, duels, ball recoveries, fouls, goalkeeper actions, substitutions, and shots.

This matters for MatchMind because different user questions may eventually require different event families.

In [ ]:
event_types = sorted({
    event["type"]["name"]
    for event in events
})

print(f"Number of unique event types: {len(event_types)}")
event_types

### 7.1 Count events by type

`Counter` provides a transparent Python-only way to count event categories.

We intentionally use it before Pandas so that the underlying operation is clear: iterate through the events, extract the event type, and count how many times each value occurs.

In [ ]:
event_type_counts = Counter(
    event["type"]["name"]
    for event in events
)

event_type_counts.most_common(15)

## 8. Inspect shot events in raw JSON

Shots are particularly useful for the first MatchMind analysis because they combine:

- categorical information, such as player, team, body part, technique, and outcome;
- temporal information, such as minute and period;
- spatial information, through the shot location;
- a model-derived value, `statsbomb_xg`.

The nested `shot` object contains shot-specific information that is not relevant to most other event types.

Important fields used in this notebook include:

| Field | Meaning |
| --- | --- |
| `player.name` | Player taking the shot. |
| `team.name` | Team taking the shot. |
| `location` | Starting position of the shot as `[x, y]`. |
| `shot.statsbomb_xg` | StatsBomb expected-goals value for the shot. |
| `shot.outcome.name` | Result of the shot, such as `Goal`, `Saved`, or another outcome. |
| `shot.body_part.name` | Body part used for the shot. |
| `shot.technique.name` | Technique used for the shot. |

### What is xG?

Expected goals, usually written **xG**, assigns a value to a shot that reflects the estimated chance of that shot becoming a goal according to the provider's model.

An xG value is useful for describing **chance quality**, not simply whether the shot happened to result in a goal.

Later in the notebook, individual xG values are summed as a descriptive measure of the total quality of chances produced. This sum should not be interpreted as the probability that a team scores a particular number of goals.

In [ ]:
shots = [
    event
    for event in events
    if event["type"]["name"] == "Shot"
]

print(f"Number of shot events: {len(shots)}")

print("\nFirst shot:")
pprint(shots[0])

### 8.1 Extract selected information from one shot

The following cell demonstrates how the same nested access pattern is used to retrieve:

- player and team;
- match time;
- spatial location;
- xG;
- outcome.

In [ ]:
shot = shots[0]

print("Player   :", shot["player"]["name"])
print("Team     :", shot["team"]["name"])
print("Minute   :", shot["minute"])
print("Location :", shot["location"])
print("xG       :", shot["shot"]["statsbomb_xg"])
print("Outcome  :", shot["shot"]["outcome"]["name"])

### 8.2 Display all raw shots in a readable format

This compact printout is still based on the raw Python dictionaries. It is useful as a final inspection step before moving to a tabular representation.

In [ ]:
for shot in shots:
    player = shot["player"]["name"]
    team = shot["team"]["name"]
    minute = shot["minute"]
    second = shot["second"]
    xg = shot["shot"]["statsbomb_xg"]
    outcome = shot["shot"]["outcome"]["name"]

    print(
        f"{minute:>3}:{second:02d} | "
        f"{team:<10} | "
        f"{player:<28} | "
        f"xG={xg:.3f} | "
        f"{outcome}"
    )

# Part II - Tabular Analysis with Pandas

The raw JSON representation is the best place to learn the data structure, but it becomes inconvenient for repeated filtering, grouping, and aggregation.

Pandas gives us a tabular representation in which:

- each row represents an event;
- each column represents an attribute;
- nested JSON keys can be flattened into column names.

For example:

```text
JSON access                         Pandas column
--------------------------------------------------------------
event["team"]["name"]               team.name
event["player"]["name"]             player.name
event["shot"]["statsbomb_xg"]       shot.statsbomb_xg
event["shot"]["outcome"]["name"]    shot.outcome.name
```

This transformation does not create new football information. It only changes the representation so that analysis becomes easier.

## 9. Normalize the event data

`pd.json_normalize()` flattens nested dictionaries into columns using dot-separated names.

Because different event types have different nested fields, many cells will be missing. For example, `shot.statsbomb_xg` only makes sense for shot events, while `pass.recipient.name` only applies to pass events.

In [ ]:
events_df = pd.json_normalize(events)

print(f"DataFrame shape: {events_df.shape}")
events_df.head()

### 9.1 Inspect the available columns

The number of columns is much larger than the small set of common event fields because event-specific nested objects have also been flattened.

Listing the columns is a useful schema-discovery step before selecting the fields needed for a specific analysis.

In [ ]:
print(f"Number of columns: {len(events_df.columns)}")

events_df.columns.tolist()

### 9.2 Build a compact event view

For general inspection, we keep only a small set of fields and rename nested column names to shorter analysis-friendly labels.

The original `events_df` remains unchanged. The renamed table is only a simplified view.

In [ ]:
basic_columns = [
    "minute",
    "second",
    "period",
    "team.name",
    "player.name",
    "type.name",
]

events_basic = events_df[basic_columns].rename(
    columns={
        "team.name": "team",
        "player.name": "player",
        "type.name": "event_type",
    }
)

events_basic.head(20)

## 10. Event distribution

`value_counts()` counts how often each value appears in a column.

For event types, it gives a quick overview of what kinds of actions dominate the match event stream. This is descriptive only: a high event count does not by itself mean that an action type was more tactically important.

In [ ]:
events_df["type.name"].value_counts().head(15)

### 10.1 Events by team

The same operation can be applied to the team column.

This count represents recorded events by team, not possession percentage. Event frequency and possession are related concepts, but they are not equivalent.

In [ ]:
events_df["team.name"].value_counts()

## 11. Build a clean shot table

The complete event DataFrame contains many columns that are irrelevant when studying shots.

We therefore:

1. filter the event stream to `Shot` events;
2. select only shot-related columns;
3. rename nested StatsBomb column names to concise working names;
4. keep `period` because the final includes a penalty shootout.

The resulting `shots_clean` table becomes our main structured representation for shot analysis.

In [ ]:
shots_df = events_df[
    events_df["type.name"] == "Shot"
].copy()

shot_columns = [
    "period",
    "minute",
    "second",
    "team.name",
    "player.name",
    "location",
    "shot.statsbomb_xg",
    "shot.outcome.name",
    "shot.body_part.name",
    "shot.technique.name",
]

shots_clean = shots_df[shot_columns].rename(
    columns={
        "team.name": "team",
        "player.name": "player",
        "shot.statsbomb_xg": "xg",
        "shot.outcome.name": "outcome",
        "shot.body_part.name": "body_part",
        "shot.technique.name": "technique",
    }
).copy()

shots_clean.head(10)

## 12. Separate match play from the penalty shootout

This distinction is essential for a professional analysis of the final.

StatsBomb uses the `period` field to identify the phase of the match:

| Period | Meaning |
| --- | --- |
| `1` | First half |
| `2` | Second half |
| `3` | First half of extra time |
| `4` | Second half of extra time |
| `5` | Penalty shootout |

A penalty taken during the shootout is recorded as a shot event, but it should not normally be mixed with shots produced during regular play or extra time when computing match shot counts and xG totals.

For that reason:

- `match_shots` contains periods 1 to 4;
- `shootout_shots` contains period 5.

The shootout data is preserved rather than deleted because it can be analyzed separately later.

In [ ]:
match_shots = shots_clean[
    shots_clean["period"] <= 4
].copy()

shootout_shots = shots_clean[
    shots_clean["period"] == 5
].copy()

print(f"Match shots (periods 1-4): {len(match_shots)}")
print(f"Penalty shootout shots (period 5): {len(shootout_shots)}")

## 13. Shots by team

A shot count answers a very specific question: how many shot events did each team produce during the match, excluding the penalty shootout?

It does **not** measure shot quality. That is why the next section considers xG separately.

In [ ]:
shots_by_team = (
    match_shots
    .groupby("team")
    .size()
    .sort_values(ascending=False)
)

shots_by_team

## 14. Total xG by team

For each team, we sum the xG values of its shots from periods 1 to 4.

This gives a compact measure of the **aggregate quality of the recorded chances**.

Important interpretation:

- shot count measures volume;
- xG adds information about chance quality;
- total xG is not the same thing as goals scored;
- total xG is not a direct probability that the team scores a specific number of goals;
- the penalty shootout is excluded from this match-level total.

In [ ]:
team_xg = (
    match_shots
    .groupby("team")["xg"]
    .sum()
    .sort_values(ascending=False)
)

team_xg

## 15. Shots and xG by player

Player-level aggregation lets us answer two different types of questions:

- **shot volume**: who attempted the most shots?
- **chance quality**: who accumulated the highest total xG?

These values can differ substantially. A player may take many low-quality shots, while another may take fewer shots from much more dangerous situations.

In [ ]:
player_shots = (
    match_shots
    .groupby("player")
    .size()
    .sort_values(ascending=False)
)

player_shots

In [ ]:
player_xg = (
    match_shots
    .groupby("player")["xg"]
    .sum()
    .sort_values(ascending=False)
)

player_xg

## 16. Goals

Within the shot table, a goal is identified when:

```text
outcome == "Goal"
```

Because `match_shots` already excludes period 5, the following table describes goals scored during the match itself, including extra time, but not successful penalty-shootout attempts.

Keeping `period`, `minute`, and `second` together is useful because the final extends beyond normal regulation time.

In [ ]:
goals_df = match_shots[
    match_shots["outcome"] == "Goal"
].copy()

goals_df[
    [
        "period",
        "minute",
        "second",
        "team",
        "player",
        "xg",
    ]
]

### 16.1 Event volume by period

The following count is performed on the full event stream, not only on shots.

It provides a quick sanity check that the dataset contains events for regular time, extra time, and the penalty shootout.

In [ ]:
events_df["period"].value_counts().sort_index()

## 17. Substitutions

Substitution events contain two player roles:

| Field | Meaning |
| --- | --- |
| `player.name` | Player leaving the pitch. |
| `substitution.replacement.name` | Player entering the pitch. |

Substitutions are particularly important for MatchMind because they provide explicit temporal anchors for later questions such as:

> How did France's attacking output change after its substitutions?

That future question can combine deterministic before/after statistics with event retrieval and an LLM-generated explanation.

In [ ]:
substitutions_df = events_df[
    events_df["type.name"] == "Substitution"
].copy()

substitution_columns = [
    "period",
    "minute",
    "second",
    "team.name",
    "player.name",
    "substitution.replacement.name",
]

substitutions_clean = substitutions_df[substitution_columns].rename(
    columns={
        "team.name": "team",
        "player.name": "player_out",
        "substitution.replacement.name": "player_in",
    }
)

substitutions_clean

### 17.1 France substitutions

Filtering by team isolates the changes that will later be useful for studying France's tactical and attacking evolution during the final.

In [ ]:
substitutions_clean[
    substitutions_clean["team"] == "France"
]

## 18. Pass events

Passes are another event family with their own nested attributes.

The selected fields describe:

| Field | Meaning |
| --- | --- |
| `player.name` | Player attempting the pass. |
| `pass.recipient.name` | Intended or recorded recipient. |
| `location` | Start position of the pass. |
| `pass.end_location` | End position of the pass. |
| `pass.outcome.name` | Explicit non-completion outcome when present. |

These fields will later support pass maps, player connections, passing networks, progression analysis, and agent tools.

In [ ]:
passes_df = events_df[
    events_df["type.name"] == "Pass"
].copy()

pass_columns = [
    "period",
    "minute",
    "second",
    "team.name",
    "player.name",
    "pass.recipient.name",
    "location",
    "pass.end_location",
    "pass.outcome.name",
]

passes_clean = passes_df[pass_columns].rename(
    columns={
        "team.name": "team",
        "player.name": "player",
        "pass.recipient.name": "recipient",
        "pass.end_location": "end_location",
        "pass.outcome.name": "outcome",
    }
)

passes_clean.head(20)

### 18.1 Pass attempts by team

This is a simple count of recorded pass events.

It should not yet be interpreted as completed passes or as possession percentage.

In [ ]:
passes_clean["team"].value_counts()

### 18.2 Pass outcomes and missing values

Missing values require careful interpretation.

In StatsBomb event data, a successful pass does not need an explicit `pass.outcome` field. Therefore, after normalization, many completed passes appear with:

```text
outcome = NaN
```

This is different from blindly assuming that every missing value in every football field means success. Missingness is field-specific and must be interpreted according to the provider's schema.

For pass events specifically:

- missing `pass.outcome` generally represents a completed pass;
- explicit outcome values describe non-standard or unsuccessful results such as an incomplete pass or a pass going out.

This convention will be handled explicitly when pass-completion metrics are added later.

In [ ]:
passes_clean["outcome"].value_counts(dropna=False)

## 19. First structured match summary

The following table combines three simple deterministic statistics:

- number of shots during match play;
- total match-play xG;
- number of recorded pass attempts.

This table is intentionally small. The objective is to demonstrate that many factual football questions can already be answered exactly from structured data, without asking an LLM to infer or calculate them.

In [ ]:
summary = pd.DataFrame({
    "shots": match_shots.groupby("team").size(),
    "total_xg": match_shots.groupby("team")["xg"].sum(),
    "passes": passes_clean.groupby("team").size(),
})

summary

## 20. First analysis questions

We can now answer several concrete questions directly from the structured data:

1. How many shots did Argentina take during the match?
2. How many shots did France take during the match?
3. Which player attempted the most shots?
4. Which player accumulated the highest total xG?
5. Which substitutions were made by France?

These questions are intentionally deterministic. Later, MatchMind should answer them by calling Python tools rather than asking an LLM to estimate values from text.

In [ ]:
argentina_shots = match_shots[
    match_shots["team"] == "Argentina"
].shape[0]

france_shots = match_shots[
    match_shots["team"] == "France"
].shape[0]

most_shots_player = player_shots.index[0]
most_shots_count = int(player_shots.iloc[0])

highest_xg_player = player_xg.index[0]
highest_xg_value = float(player_xg.iloc[0])

print(f"Argentina shots: {argentina_shots}")
print(f"France shots: {france_shots}")
print(f"Most shots: {most_shots_player} ({most_shots_count})")
print(f"Highest total xG: {highest_xg_player} ({highest_xg_value:.3f})")

print("\nFrance substitutions:")
display(
    substitutions_clean[
        substitutions_clean["team"] == "France"
    ]
)

### Design implication for MatchMind

At this point, an important architectural distinction becomes visible.

Some questions are best answered by **structured tools**:

```text
"How many shots did France take?"
"Who had the highest xG?"
"Which players were substituted?"
```

These can be computed deterministically from the event data.

Other questions will eventually require **retrieval and language generation**:

```text
"Describe how France's attacking threat changed after the substitutions."
"Summarize the main attacking phases of the second half."
```

A future MatchMind agent should therefore learn to route a user request to the appropriate capability instead of using an LLM for every task.

# Part III - Spatial Analysis and Shot Maps

Football event data is not only temporal and categorical. Many events also contain spatial coordinates.

This section introduces the StatsBomb coordinate system and uses it to build the first MatchMind visualization.

## 21. StatsBomb pitch coordinates

StatsBomb represents locations in a standardized **120 by 80 analysis coordinate space**.

For the event data used here:

- the horizontal coordinate is `x`;
- `x` ranges approximately from `0` to `120`;
- the vertical coordinate is `y`;
- `y` ranges approximately from `0` to `80`;
- an event `location` is represented as `[x, y]`.

A simplified view is:

```text
y = 0
(0, 0) --------------------------------------- (120, 0)
   |                                             |
   |                                             |
   |                                             |
   |                                             |
(0, 80) -------------------------------------- (120, 80)
y = 80

x = 0                                         x = 120
```

For attacking events in this representation, the attacking direction is standardized so that shots are oriented toward the opponent goal near `x = 120`. This makes attacking actions from both teams directly comparable.

### Important interpretation

The 120 by 80 system is an **analysis coordinate system**. It should not be interpreted as a claim that every physical stadium pitch has exactly those dimensions.

For a shot:

```python
location = [108.2, 42.7]
```

means:

```text
x = 108.2
y = 42.7
```

A larger `x` generally places the shot closer to the opponent goal line.

This spatial representation is the basis for future MatchMind features such as:

- shot maps;
- pass maps;
- passing networks;
- spatial retrieval;
- heatmaps;
- player-location analysis.

## 22. Visualization imports

Matplotlib is used directly rather than a football-specific plotting package.

This is intentional for the first visualization: drawing the pitch ourselves makes the coordinate system and plot geometry explicit before introducing higher-level libraries.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

## 23. Inspect raw shot locations

Before extracting coordinates into separate columns, we inspect several `[x, y]` values directly.

This verifies that the spatial information is stored inside the shot table exactly as expected.

In [ ]:
shots_clean[["player", "team", "location"]].head(10)

## 24. Extract `x` and `y`

The `location` column contains a two-element list for each shot.

Splitting it into explicit `x` and `y` columns makes the spatial data easier to:

- filter;
- plot;
- aggregate;
- pass to future visualization functions.

Because a shot map cannot be built correctly without valid coordinates, we also perform a small validation check.

In [ ]:
valid_locations = shots_clean["location"].apply(
    lambda location: isinstance(location, list) and len(location) >= 2
)

if not valid_locations.all():
    raise ValueError("At least one shot does not contain a valid [x, y] location.")

shots_clean["x"] = shots_clean["location"].apply(
    lambda location: location[0]
)

shots_clean["y"] = shots_clean["location"].apply(
    lambda location: location[1]
)

shots_clean[
    ["player", "team", "x", "y", "xg", "outcome"]
].head()

## 25. Recreate the match-play and shootout subsets

The shot table now contains two additional columns, `x` and `y`.

We recreate the two subsets so that the new spatial columns are also present in both DataFrames:

- `match_shots`: periods 1 to 4;
- `shootout_shots`: period 5.

All visualizations below focus on match play and extra time, not the shootout.

In [ ]:
match_shots = shots_clean[
    shots_clean["period"] <= 4
].copy()

shootout_shots = shots_clean[
    shots_clean["period"] == 5
].copy()

print(f"Match shots: {len(match_shots)}")
print(f"Penalty shootout shots: {len(shootout_shots)}")

## 26. First spatial scatter plot

Before drawing a football pitch, it is useful to plot the coordinates directly.

This simple scatter plot answers one question only:

> Where are the recorded shots located in the StatsBomb coordinate system?

No tactical styling is added yet. This makes it easier to confirm that the extracted coordinates behave as expected.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    match_shots["x"],
    match_shots["y"]
)

plt.xlim(0, 120)
plt.ylim(0, 80)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Shot Locations - Argentina vs France")

plt.show()

## 27. Encode shot quality with marker size

A spatial point tells us **where** a shot occurred, but not how dangerous the chance was.

We can use marker size to encode xG:

```text
lower xG  -> smaller marker
higher xG -> larger marker
```

The multiplication by `1000` is only a visual scaling factor for Matplotlib. It does not change the original xG values or their interpretation.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    match_shots["x"],
    match_shots["y"],
    s=match_shots["xg"] * 1000
)

plt.xlim(0, 120)
plt.ylim(0, 80)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Shot Locations Sized by xG")

plt.show()

## 28. Draw the pitch manually

The next function constructs a simplified pitch from Matplotlib primitives.

It uses:

- lines for the pitch boundaries and halfway line;
- rectangles for the penalty areas and six-yard boxes;
- a circle for the centre circle;
- points for the centre and penalty spots;
- thicker short lines for the goals.

The geometric values are expressed directly in the same 120 by 80 coordinate system as the StatsBomb event locations. This ensures that event coordinates can be overlaid without an additional transformation.

The function only draws the pitch. It does not contain football data, which makes it reusable for other spatial event plots.

In [ ]:
def draw_pitch(ax):
    # Pitch boundaries
    ax.plot([0, 0], [0, 80])
    ax.plot([0, 120], [80, 80])
    ax.plot([120, 120], [80, 0])
    ax.plot([120, 0], [0, 0])

    # Halfway line
    ax.plot([60, 60], [0, 80])

    # Centre circle
    centre_circle = Circle(
        (60, 40),
        radius=10,
        fill=False
    )
    ax.add_patch(centre_circle)

    # Centre spot
    ax.scatter(60, 40, s=10)

    # Left penalty area
    ax.add_patch(
        Rectangle(
            (0, 18),
            18,
            44,
            fill=False
        )
    )

    # Right penalty area
    ax.add_patch(
        Rectangle(
            (102, 18),
            18,
            44,
            fill=False
        )
    )

    # Left six-yard box
    ax.add_patch(
        Rectangle(
            (0, 30),
            6,
            20,
            fill=False
        )
    )

    # Right six-yard box
    ax.add_patch(
        Rectangle(
            (114, 30),
            6,
            20,
            fill=False
        )
    )

    # Penalty spots
    ax.scatter([12, 108], [40, 40], s=10)

    # Goals
    ax.plot([0, 0], [36, 44], linewidth=4)
    ax.plot([120, 120], [36, 44], linewidth=4)

    ax.set_xlim(-2, 122)
    ax.set_ylim(-2, 82)

    ax.set_aspect("equal")
    ax.axis("off")

### 28.1 Validate the pitch representation

Before adding any match data, we plot the empty pitch by itself.

This separation is useful for debugging. If a later shot map looks wrong, we can distinguish a pitch-drawing problem from a coordinate or filtering problem.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

draw_pitch(ax)

ax.set_title(
    "StatsBomb Pitch Coordinate System",
    fontsize=16
)

plt.show()

## 29. Overlay the match shots

We now combine the two components developed independently:

```text
pitch geometry
      +
shot coordinates
      +
xG marker size
      =
shot map
```

At this stage, all shots use the same marker style. The objective is to verify spatial alignment.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

draw_pitch(ax)

ax.scatter(
    match_shots["x"],
    match_shots["y"],
    s=match_shots["xg"] * 1000,
    alpha=0.6
)

ax.set_title(
    "Argentina vs France - Shot Map",
    fontsize=16
)

plt.show()

## 30. Distinguish the two teams

Separating the DataFrame by team allows Matplotlib to assign a different visual series and legend entry to each side.

The underlying coordinates remain unchanged. This is only a visual encoding of team membership.

In [ ]:
argentina_shots = match_shots[
    match_shots["team"] == "Argentina"
]

france_shots = match_shots[
    match_shots["team"] == "France"
]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

draw_pitch(ax)

ax.scatter(
    argentina_shots["x"],
    argentina_shots["y"],
    s=argentina_shots["xg"] * 1000,
    alpha=0.7,
    label="Argentina"
)

ax.scatter(
    france_shots["x"],
    france_shots["y"],
    s=france_shots["xg"] * 1000,
    alpha=0.7,
    label="France"
)

ax.legend()

ax.set_title(
    "Argentina vs France - 2022 World Cup Final",
    fontsize=16
)

plt.show()

## 31. Identify goals explicitly

A shot map becomes easier to interpret when goals are visually distinguishable from non-goal attempts.

We create a Boolean field:

```text
is_goal = True  -> shot outcome is Goal
is_goal = False -> any other shot outcome
```

This field does not replace the original `outcome` column. It is a convenient derived feature for plotting.

In [ ]:
match_shots["is_goal"] = (
    match_shots["outcome"] == "Goal"
)

In [ ]:
match_shots[
    [
        "minute",
        "player",
        "team",
        "xg",
        "is_goal"
    ]
].head()

### 31.1 Use a different marker for goals

In the following plot:

- circles represent non-goal shots;
- stars represent goals;
- marker size continues to represent xG.

This allows three dimensions of information to be read simultaneously:

```text
position -> where the shot was taken
size     -> estimated chance quality
shape    -> whether the shot became a goal
```

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

draw_pitch(ax)

for team, team_shots in match_shots.groupby("team"):

    non_goals = team_shots[
        ~team_shots["is_goal"]
    ]

    goals = team_shots[
        team_shots["is_goal"]
    ]

    ax.scatter(
        non_goals["x"],
        non_goals["y"],
        s=non_goals["xg"] * 1000,
        alpha=0.5,
        label=f"{team} shots"
    )

    ax.scatter(
        goals["x"],
        goals["y"],
        s=goals["xg"] * 1000,
        marker="*",
        label=f"{team} goals"
    )

ax.legend()

ax.set_title(
    "Argentina vs France - Shot Map",
    fontsize=16
)

plt.show()

## 32. Annotate the goals

Goal annotations make the visualization more interpretable without requiring the reader to cross-reference another table.

The first cell checks the exact goal coordinates and players. The second adds the player name and match minute next to each goal marker.

In [ ]:
goals = match_shots[
    match_shots["is_goal"]
]

for _, goal in goals.iterrows():
    print(
        goal["minute"],
        goal["player"],
        goal["x"],
        goal["y"]
    )

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

draw_pitch(ax)

for team, team_shots in match_shots.groupby("team"):

    ax.scatter(
        team_shots["x"],
        team_shots["y"],
        s=team_shots["xg"] * 1000,
        alpha=0.55,
        label=team
    )

for _, goal in goals.iterrows():

    ax.scatter(
        goal["x"],
        goal["y"],
        s=goal["xg"] * 1000,
        marker="*"
    )

    ax.text(
        goal["x"] - 2,
        goal["y"] - 2,
        f"{goal['player']} {goal['minute']}'",
        fontsize=8
    )

ax.legend()

ax.set_title(
    "Argentina vs France - Shot Map",
    fontsize=16
)

plt.show()

## 33. Focus on the attacking half

Most shot information is concentrated close to the opponent goal.

Restricting the horizontal view to the attacking half:

```python
ax.set_xlim(60, 122)
```

does not filter or modify the data. It only changes the visible plot area.

This is useful when the analytical question concerns shot locations rather than full-pitch context.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

draw_pitch(ax)

for team, team_shots in match_shots.groupby("team"):

    ax.scatter(
        team_shots["x"],
        team_shots["y"],
        s=team_shots["xg"] * 1000,
        alpha=0.6,
        label=team
    )

ax.set_xlim(60, 122)
ax.set_ylim(-2, 82)

ax.legend()

ax.set_title(
    "Argentina vs France - Attacking Half Shot Map",
    fontsize=16
)

plt.show()

# Part IV - Interpretation and Project Takeaways

## 34. What this notebook has established

We now have a complete first data-analysis path:

```text
StatsBomb Open Data
        |
        v
raw JSON
        |
        v
Python lists and dictionaries
        |
        v
nested event schema
        |
        v
Pandas normalization
        |
        +------------------+
        |                  |
        v                  v
structured statistics   spatial coordinates
        |                  |
        v                  v
shots / xG / passes     shot maps
```

The important result is not only the final visualization. The notebook establishes a clear understanding of how football information is represented and transformed.

## 35. Key analytical decisions

Several decisions in this notebook are deliberate:

### Raw data before abstraction

We inspect the JSON before using convenience libraries so that nested event structures are understood rather than hidden.

### Match play separated from the shootout

Periods 1 to 4 are used for match shot and xG analysis. Period 5 is preserved separately. This prevents penalty-shootout attempts from contaminating normal match-performance statistics.

### Deterministic computation before LLM generation

Questions involving exact counts, sums, filters, and coordinates are solved with Python. An LLM should not be asked to guess information that can be computed exactly.

### Visualization encodes multiple variables

The shot map represents:

- `x`, `y` -> spatial location;
- marker size -> xG;
- marker shape -> goal or non-goal;
- plotting series -> team.

This makes the visualization an analytical object rather than a decorative chart.

## 36. Limitations of the current analysis

This notebook is intentionally introductory. It does not yet:

- compute possession-adjusted metrics;
- model tactical formations;
- calculate pass-completion percentages;
- distinguish open-play and set-piece xG in the summary;
- use StatsBomb 360 freeze-frame information;
- analyze defensive structure;
- evaluate sequences or possessions as semantic units;
- use embeddings, a vector database, RAG, or an LLM.

These are future layers of MatchMind, not missing requirements for this first notebook.

## 37. Why this matters for the future AI agent

The current notebook already suggests two different families of future MatchMind tools.

### Structured analytics tools

Examples:

```text
get_team_shots()
get_player_shots()
get_team_xg()
get_substitutions()
plot_shot_map()
```

These should operate directly on structured football data.

### Semantic and RAG tools

Later, MatchMind will transform richer event sequences into text representations that can support questions such as:

```text
"Describe France's attacking phases after the 70th minute."
"What changed after the first French substitutions?"
"Summarize the sequence leading to a specific chance."
```

Those questions require context retrieval and language synthesis rather than only numerical aggregation.

The long-term architecture will therefore combine both approaches:

```text
User question
      |
      v
Intent / tool selection
      |
      +----------------------+----------------------+
      |                                             |
      v                                             v
Structured football tools                    Retrieval / RAG
      |                                             |
      +----------------------+----------------------+
                             |
                             v
                      Final explanation
```

## 38. Next project step

The next step is to move the stable, reusable logic out of this exploratory notebook and into the project source code.

Candidate functions include:

```python
load_events(...)
get_match_shots(...)
get_player_shots(...)
get_team_xg(...)
get_substitutions(...)
plot_shot_map(...)
```

The notebook remains useful for experimentation and explanation, while `src/` becomes the location for reusable MatchMind components.

After that foundation is clean, the project can begin the transition toward textual event representations, chunking, embeddings, semantic retrieval, and eventually RAG.